# 06 — The KV Cache and Inference Speed

**Companion to [Chapter 08](../08-inference-and-efficiency.md).**

Naive generation recomputes the entire prefix for every new token. The KV cache
eliminates that — turning O(n^2) total work into O(n).

You will implement it, measure the speedup, and compute the memory it costs
(which is why GQA exists).

In [1]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


## 1. The waste, made visible

To generate token 101, naive generation feeds all 100 previous tokens through the
model and discards 100 of the 101 predictions.

But because of causal masking, **the keys and values of tokens 1-100 do not change
when token 101 arrives**. Recomputing them is pure waste.

In [2]:
total_naive = sum(range(1, 201))
total_cached = 200
print(f"generating 200 tokens:")
print(f"  naive  : {total_naive:,} token-forward-passes  (recompute the prefix every step)")
print(f"  cached : {total_cached:,} token-forward-passes")
print(f"  ratio  : {total_naive / total_cached:.1f}x more work without a cache")
print()
print("And it gets worse with length: the gap grows linearly in sequence length.")

generating 200 tokens:
  naive  : 20,100 token-forward-passes  (recompute the prefix every step)
  cached : 200 token-forward-passes
  ratio  : 100.5x more work without a cache

And it gets worse with length: the gap grows linearly in sequence length.


## 2. Attention with a KV cache

The change is small: keep `k` and `v` between calls and concatenate the new token's.

In [3]:
class CachedAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, cache=None):
        b, seq, d = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q, k, v = [t.view(b, seq, self.n_heads, self.head_dim).transpose(1, 2)
                   for t in (q, k, v)]

        if cache is not None:
            past_k, past_v = cache
            k = torch.cat([past_k, k], dim=2)      # append along the sequence axis
            v = torch.cat([past_v, v], dim=2)
        new_cache = (k, v)

        # causal masking only needed when processing >1 token at once (prefill)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=(seq > 1))
        out = out.transpose(1, 2).contiguous().view(b, seq, d)
        return self.proj(out), new_cache


attn = CachedAttention(256, 8).to(device)

# PREFILL: the whole prompt at once
prompt = torch.randn(1, 10, 256, device=device)
out, cache = attn(prompt)
print("after prefill (10 tokens) -> cached K:", tuple(cache[0].shape))

# DECODE: one token at a time, reusing the cache
for step in range(3):
    out, cache = attn(torch.randn(1, 1, 256, device=device), cache)
    print(f"after decode step {step + 1}      -> cached K:", tuple(cache[0].shape))

print("\nThe cache grows by exactly one position per generated token.")

after prefill (10 tokens) -> cached K: (1, 8, 10, 32)
after decode step 1      -> cached K: (1, 8, 11, 32)
after decode step 2      -> cached K: (1, 8, 12, 32)
after decode step 3      -> cached K: (1, 8, 13, 32)

The cache grows by exactly one position per generated token.


## 3. Correctness check

Cached and uncached generation must produce identical results.

In [4]:
torch.manual_seed(7)
attn = CachedAttention(64, 4).to(device).eval()
x_full = torch.randn(1, 8, 64, device=device)

with torch.no_grad():
    # uncached: process everything at once
    full_out, _ = attn(x_full)

    # cached: one token at a time
    cached_outs, cache = [], None
    for i in range(8):
        o, cache = attn(x_full[:, i:i + 1], cache)
        cached_outs.append(o)
    incremental = torch.cat(cached_outs, dim=1)

print("identical:", torch.allclose(full_out, incremental, atol=1e-5))
print("max difference:", (full_out - incremental).abs().max().item())
print("\nSame math. The cache is a pure optimization, not an approximation.")

identical: True
max difference: 3.2782554626464844e-07

Same math. The cache is a pure optimization, not an approximation.


## 4. Measure the speedup

A small stack of cached-attention layers, generating with and without the cache.

In [5]:
class TinyStack(nn.Module):
    def __init__(self, d_model=256, n_heads=8, n_layers=4):
        super().__init__()
        self.layers = nn.ModuleList([CachedAttention(d_model, n_heads) for _ in range(n_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])

    def forward(self, x, caches=None):
        new_caches = []
        for i, (layer, norm) in enumerate(zip(self.layers, self.norms)):
            c = caches[i] if caches is not None else None
            out, nc = layer(norm(x), c)
            x = x + out
            new_caches.append(nc)
        return x, new_caches


@torch.no_grad()
def generate_naive(model, prompt, n_new):
    x = prompt
    for _ in range(n_new):
        out, _ = model(x)                       # recompute EVERYTHING every step
        x = torch.cat([x, out[:, -1:, :]], dim=1)
    return x


@torch.no_grad()
def generate_cached(model, prompt, n_new):
    out, caches = model(prompt)                 # prefill once
    x = prompt
    nxt = out[:, -1:, :]
    for _ in range(n_new):
        x = torch.cat([x, nxt], dim=1)
        out, caches = model(nxt, caches)        # one token at a time
        nxt = out[:, -1:, :]
    return x


torch.manual_seed(0)
model = TinyStack().to(device).eval()
prompt = torch.randn(1, 16, 256, device=device)

print(f"{'new tokens':>11} {'naive (s)':>11} {'cached (s)':>11} {'speedup':>9}")
for n_new in [32, 64, 128]:
    t0 = time.time(); generate_naive(model, prompt, n_new); t_naive = time.time() - t0
    t0 = time.time(); generate_cached(model, prompt, n_new); t_cached = time.time() - t0
    print(f"{n_new:>11} {t_naive:>11.3f} {t_cached:>11.3f} {t_naive / t_cached:>8.1f}x")

print("\nThe speedup GROWS with sequence length, because the naive version's")
print("wasted work grows while the cached version's does not.")

 new tokens   naive (s)  cached (s)   speedup
         32       0.036       0.016      2.3x


         64       0.071       0.033      2.2x

        128       0.204       0.066      3.1x

The speedup GROWS with sequence length, because the naive version's
wasted work grows while the cached version's does not.


## 5. What the cache costs

```
KV cache bytes = 2 (K and V)
               x n_layers
               x n_kv_heads
               x head_dim
               x seq_len
               x batch_size
               x bytes_per_value
```

This is the number that decides how many users fit on a GPU.

In [6]:
def kv_cache_gb(n_layers, n_kv_heads, head_dim, seq_len, batch, bytes_per=2):
    return 2 * n_layers * n_kv_heads * head_dim * seq_len * batch * bytes_per / 1e9


rows = [
    ("GPT-2 124M, 1k ctx,  1 seq",   12, 12,  64,   1024,  1),
    ("7B MHA,     4k ctx,  1 seq",   32, 32, 128,   4096,  1),
    ("7B MHA,    32k ctx,  1 seq",   32, 32, 128,  32768,  1),
    ("7B GQA-8,  32k ctx,  1 seq",   32,  8, 128,  32768,  1),
    ("7B GQA-8,  32k ctx, 32 seq",   32,  8, 128,  32768, 32),
    ("70B GQA-8,128k ctx,  1 seq",   80,  8, 128, 131072,  1),
]
print(f"{'configuration':<28} {'KV cache':>12}")
print("-" * 42)
for label, L, H, hd, s, b in rows:
    print(f"{label:<28} {kv_cache_gb(L, H, hd, s, b):>9.2f} GB")

print("\nCompare: a 7B model's WEIGHTS are ~14 GB at bf16.")
print("At 32k context with full MHA, the cache alone exceeds the weights.")
print("GQA with 8 KV heads cuts it 4x. That is the entire reason GQA exists.")

configuration                    KV cache
------------------------------------------
GPT-2 124M, 1k ctx,  1 seq        0.04 GB
7B MHA,     4k ctx,  1 seq        2.15 GB
7B MHA,    32k ctx,  1 seq       17.18 GB
7B GQA-8,  32k ctx,  1 seq        4.29 GB
7B GQA-8,  32k ctx, 32 seq      137.44 GB
70B GQA-8,128k ctx,  1 seq       42.95 GB

Compare: a 7B model's WEIGHTS are ~14 GB at bf16.
At 32k context with full MHA, the cache alone exceeds the weights.
GQA with 8 KV heads cuts it 4x. That is the entire reason GQA exists.


### GQA savings, directly

In [7]:
base = kv_cache_gb(32, 32, 128, 32768, 8)
print(f"{'scheme':<16} {'KV heads':>9} {'cache':>10} {'saving':>9}")
for label, kv_heads in [("MHA", 32), ("GQA-8", 8), ("GQA-4", 4), ("MQA", 1)]:
    c = kv_cache_gb(32, kv_heads, 128, 32768, 8)
    print(f"{label:<16} {kv_heads:>9} {c:>7.2f} GB {base / c:>8.1f}x")

scheme            KV heads      cache    saving
MHA                     32  137.44 GB      1.0x
GQA-8                    8   34.36 GB      4.0x
GQA-4                    4   17.18 GB      8.0x
MQA                      1    4.29 GB     32.0x


## 6. Prefill vs decode: two different workloads

```
PREFILL                        DECODE
whole prompt at once           one token at a time
parallel across positions      strictly sequential
COMPUTE-bound                  MEMORY-BANDWIDTH-bound
sets time-to-first-token       sets tokens/second
```

Decode reads **every model weight from memory** to produce a single token. That
one fact explains most of production serving.

In [8]:
def decode_speed(params_b, bytes_per_param, bandwidth_tbs=2.0):
    bytes_per_token = params_b * 1e9 * bytes_per_param
    return bandwidth_tbs * 1e12 / bytes_per_token


print("Theoretical single-sequence decode speed (2 TB/s GPU):\n")
print(f"{'model':>8} {'bf16':>14} {'int8':>14} {'int4':>14}")
for p in [7, 13, 70]:
    speeds = [decode_speed(p, b) for b in (2, 1, 0.5)]
    print(f"{p:>6}B " + "".join(f"{s:>11.0f} tok/s" for s in speeds))

print("\nThree consequences:")
print("  1. Quantization speeds up DECODING - fewer bytes to move, not fewer FLOPs")
print("  2. Batching is nearly free per user - weights are read once for the whole batch")
print("  3. Prefill and decode have different bottlenecks, so they schedule differently")

Theoretical single-sequence decode speed (2 TB/s GPU):

   model           bf16           int8           int4
     7B         143 tok/s        286 tok/s        571 tok/s
    13B          77 tok/s        154 tok/s        308 tok/s
    70B          14 tok/s         29 tok/s         57 tok/s

Three consequences:
  1. Quantization speeds up DECODING - fewer bytes to move, not fewer FLOPs
  2. Batching is nearly free per user - weights are read once for the whole batch
  3. Prefill and decode have different bottlenecks, so they schedule differently


## 7. A capacity-planning calculator

This back-of-envelope is genuinely how serving capacity planning starts.

In [9]:
def serving_estimate(params_b, seq_len, batch, n_layers, n_kv_heads, head_dim,
                     bytes_per_param=2, bandwidth_tbs=2.0, gpu_mem_gb=80):
    weights = params_b * 1e9 * bytes_per_param / 1e9
    kv = kv_cache_gb(n_layers, n_kv_heads, head_dim, seq_len, batch)
    tps = decode_speed(params_b, bytes_per_param, bandwidth_tbs)
    fits = weights + kv < gpu_mem_gb * 0.9        # leave headroom for activations

    print(f"  weights          {weights:7.1f} GB")
    print(f"  KV cache         {kv:7.1f} GB   (batch={batch}, ctx={seq_len:,})")
    print(f"  total            {weights + kv:7.1f} GB / {gpu_mem_gb} GB   "
          f"{'FITS' if fits else 'DOES NOT FIT'}")
    print(f"  decode speed     ~{tps:6.0f} tok/s per sequence")
    print(f"  aggregate        ~{tps * batch:6.0f} tok/s across the batch")


print("7B, GQA-8, 8k context, batch 32, bf16:")
serving_estimate(7, 8192, 32, 32, 8, 128)

print("\nSame model quantized to int4:")
serving_estimate(7, 8192, 32, 32, 8, 128, bytes_per_param=0.5)

7B, GQA-8, 8k context, batch 32, bf16:
  weights             14.0 GB
  KV cache            34.4 GB   (batch=32, ctx=8,192)
  total               48.4 GB / 80 GB   FITS
  decode speed     ~   143 tok/s per sequence
  aggregate        ~  4571 tok/s across the batch

Same model quantized to int4:
  weights              3.5 GB
  KV cache            34.4 GB   (batch=32, ctx=8,192)
  total               37.9 GB / 80 GB   FITS
  decode speed     ~   571 tok/s per sequence
  aggregate        ~ 18286 tok/s across the batch


## Exercises

1. Change `bytes_per_param` to 0.5 in the calculator and explain both effects (memory *and* speed).
2. At what batch size does a 7B GQA-8 model at 32k context stop fitting in 80 GB?
3. Implement a **sliding-window** cache that keeps only the last 1024 positions. Measure the memory saving and think about what capability it costs.

**Next:** [07 — LoRA Finetuning](07_lora_finetuning.ipynb)